# Contrastive Learning for UCI HAR

This notebook demonstrates contrastive learning for the UCI HAR dataset, including data loading, augmentation, model definition, pre-training, linear evaluation, and fine-tuning.

## 0. Install and Imports

In [ ]:
# Uncomment the following line to install necessary packages
# !pip install torch torchvision scikit-learn matplotlib tqdm

import os, urllib.request, zipfile, io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline # Required for magnitude_warp and time_warp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


This cell handles the initial setup. It imports all necessary libraries such as `torch` for deep learning, `numpy` for numerical operations, `sklearn` for machine learning utilities, `matplotlib` for plotting, and `tqdm` for progress bars. It also checks for the availability of a GPU (`cuda`) and sets the `DEVICE` variable accordingly, printing which device will be used for computations.

## 1. Download & Load UCI HAR Dataset

In [ ]:
def download_uci_har(data_dir="UCI_HAR"):
    url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
           "00240/UCI%20HAR%20Dataset.zip")
    if not os.path.exists(data_dir):
        print("Downloading UCI HAR Dataset...")
        # Explicitly create and use an unverified SSL context to handle certificate issues
        import ssl
        context = ssl._create_unverified_context()
        with urllib.request.urlopen(url, context=context) as r:
            with zipfile.ZipFile(io.BytesIO(r.read())) as zf:
                zf.extractall(".")
        os.rename("UCI HAR Dataset", data_dir)
        print("Done.")
    return data_dir

def load_signals(base, split):
    """Load 9 raw inertial signals → (N, 128, 9)"""
    signals_dir = os.path.join(base, split, "Inertial Signals")
    files = sorted([f for f in os.listdir(signals_dir) if f.endswith(".txt")])
    channels = []
    for f in files:
        data = np.loadtxt(os.path.join(signals_dir, f))  # (N, 128)
        channels.append(data)
    return np.stack(channels, axis=-1).astype(np.float32)   # (N, 128, 9)

def load_labels(base, split):
    path = os.path.join(base, split, f"y_{split}.txt")
    return np.loadtxt(path).astype(int) - 1                  # 0-indexed

data_dir = download_uci_har()

X_train_raw = load_signals(data_dir, "train")   # (7352, 128, 9)
y_train     = load_labels(data_dir,  "train")
X_test_raw  = load_signals(data_dir, "test")    # (2947, 128, 9)
y_test      = load_labels(data_dir,  "test")

This cell defines functions to download and load the UCI HAR (Human Activity Recognition) dataset.  
*   `download_uci_har`: Downloads the dataset from a specified URL, extracts it, and renames the directory.  
*   `load_signals`: Reads the raw inertial signals (accelerometer and gyroscope data) from text files for a given dataset split (train/test) and stacks them into a NumPy array of shape (N, 128, 9), where N is the number of samples, 128 is the time steps, and 9 are the channels.  
*   `load_labels`: Loads the activity labels for a given dataset split, adjusting them to be 0-indexed.  
The cell then calls these functions to download the data and load the raw training and testing signals and labels into `X_train_raw`, `y_train`, `X_test_raw`, and `y_test` respectively.

## 2. Normalize and Prepare Data

In [ ]:
# Normalize each channel (fit on train, apply to both)
N_tr, T, C = X_train_raw.shape
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train_raw.reshape(-1, C)).reshape(N_tr, T, C)
X_test_norm  = scaler.transform(X_test_raw.reshape(-1, C)).reshape(-1, T, C)

# (N, 9, 128)  →  channels-first for Conv1d
X_train = torch.tensor(X_train_norm.transpose(0, 2, 1), dtype=torch.float32)
X_test  = torch.tensor(X_test_norm.transpose(0, 2, 1),  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Classes: {np.unique(y_train)}")

X_train: torch.Size([7352, 9, 128]), X_test: torch.Size([2947, 9, 128])
Classes: [0 1 2 3 4 5]


This cell preprocesses the raw sensor data.  
*   It first calculates the dimensions of the training data (`N_tr` samples, `T` time steps, `C` channels).  
*   A `StandardScaler` from scikit-learn is used to normalize each channel of the sensor data. It's `fit` on the training data and then used to `transform` both the training and test data, ensuring consistency. The data is reshaped during scaling to treat each channel's time series independently.  
*   The normalized NumPy arrays are then converted to PyTorch tensors. Crucially, the data is transposed from (N, T, C) to (N, C, T) (or `(batch_size, channels, sequence_length)`) to match the expected input format for 1D Convolutional Neural Networks (`Conv1d`) in PyTorch.  
*   The labels `y_train` and `y_test` are also converted to PyTorch tensors of type `long`.  
Finally, it prints the shapes of the processed `X_train` and `X_test` tensors and the unique classes in `y_train` to verify the data preparation.

## 3. Time-Series Augmentations

In [ ]:
def jitter(x, sigma=0.05):
    return x + torch.randn_like(x) * sigma

def scaling(x, sigma=0.15):
    scale = 1.0 + torch.randn(x.size(0), 1, device=x.device) * sigma
    return x * scale

def time_shift(x, max_shift=20):
    shift = np.random.randint(-max_shift, max_shift)
    return torch.roll(x, shift, dims=-1)

def magnitude_warp(x, knots=4, sigma=0.15):
    """Smooth random magnitude scaling via cubic interpolation."""
    t_orig = np.linspace(0, 1, x.shape[-1])
    t_knot = np.linspace(0, 1, knots + 2)
    warped = []
    for xi in x.cpu().numpy():
        scales = np.random.normal(1.0, sigma, size=len(t_knot))
        cs = CubicSpline(t_knot, scales)
        w  = torch.tensor(cs(t_orig), dtype=torch.float32)
        warped.append(torch.tensor(xi) * w)
    return torch.stack(warped).to(x.device)

def time_warp(x, sigma=0.2, knots=4):
    """Warp the time axis smoothly."""
    T_len = x.shape[-1]
    t_orig = np.arange(T_len, dtype=np.float32)
    t_knot = np.linspace(0, T_len - 1, knots + 2)
    warped = []
    for xi in x.cpu().numpy():
        deltas = np.random.normal(0, sigma * T_len, size=len(t_knot))
        deltas[[0, -1]] = 0
        new_t = np.clip(t_knot + deltas, 0, T_len - 1)
        cs    = CubicSpline(new_t, t_knot)
        new_x = np.array([np.interp(cs(np.linspace(0, T_len-1, T_len)), t_orig, xi[c])
                           for c in range(xi.shape[0])])
        warped.append(torch.tensor(new_x, dtype=torch.float32))
    return torch.stack(warped).to(x.device)

def window_slice(x, reduce=0.9):
    T_len = x.shape[-1]
    start = np.random.randint(0, int(T_len * (1 - reduce)) + 1)
    end   = start + int(T_len * reduce)
    sliced = x[:, :, start:end]
    return F.interpolate(sliced, size=T_len, mode='linear', align_corners=False)

def window_warp(x, window_ratio=0.3, scales=(0.5, 2.0)):
    T_len = x.shape[-1]
    wlen  = int(T_len * window_ratio)
    start = np.random.randint(0, T_len - wlen)
    scale = np.random.uniform(*scales)
    head  = x[:, :, :start]
    mid   = F.interpolate(x[:, :, start:start+wlen],
                          size=int(wlen * scale), mode='linear', align_corners=False)
    tail  = x[:, :, start+wlen:]
    cat   = torch.cat([head, mid, tail], dim=-1)
    return F.interpolate(cat, size=T_len, mode='linear', align_corners=False)

def dropout_segments(x, p=0.1, n_segs=3):
    """Zero out random segments."""
    out = x.clone()
    T_len = x.shape[-1]
    seg_len = max(1, int(T_len * p))
    for _ in range(n_segs):
        start = np.random.randint(0, T_len - seg_len)
        out[:, :, start:start+seg_len] = 0
    return out

# Strong augmentation pipeline: randomly apply 2–4 transforms
AUG_LIST = [jitter, scaling, time_shift, window_slice, window_warp, dropout_segments]

def strong_augment(x):
    """Apply 2-4 randomly chosen augmentations."""
    chosen = np.random.choice(len(AUG_LIST), size=np.random.randint(2, 5), replace=False)
    for idx in chosen:
        x = AUG_LIST[idx](x)
    return x

This cell defines various time-series data augmentation techniques crucial for contrastive learning. These functions introduce variations into the input data without changing its underlying meaning, helping the model learn more robust representations.  
*   `jitter`: Adds random noise to the signal.  
*   `scaling`: Multiplies the signal by a random scaling factor.  
*   `time_shift`: Shifts the signal along the time axis.  
*   `magnitude_warp`: Smoothly warps the signal's magnitude using cubic interpolation.  
*   `time_warp`: Smoothly warps the signal's time axis using cubic interpolation.  
*   `window_slice`: Extracts a random sub-segment of the signal and then interpolates it back to the original length.  
*   `window_warp`: Stretches or compresses a random window of the signal and then interpolates the entire signal back to the original length.  
*   `dropout_segments`: Zeros out random segments of the signal.  
  
`AUG_LIST` is a list containing all these augmentation functions.  
`strong_augment`: This function takes an input `x` and applies 2 to 4 randomly chosen augmentations from `AUG_LIST` to it. This creates a strongly augmented view of the original data, which is essential for contrastive learning where the model learns by distinguishing between augmented views of the same data point versus different data points.

## 4. Contrastive Dataset and DataLoader

In [ ]:
class ContrastiveHARDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)          # (1, 9, 128)
        x1 = strong_augment(x).squeeze(0)
        x2 = strong_augment(x).squeeze(0)
        return x1, x2

BATCH_SIZE = 256
train_cl_dataset = ContrastiveHARDataset(X_train)
cl_loader = DataLoader(train_cl_dataset, batch_size=BATCH_SIZE,
                       shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

This cell defines a custom PyTorch `Dataset` and `DataLoader` for contrastive learning.  
*   `ContrastiveHARDataset`: This class inherits from `torch.utils.data.Dataset`.  
    *   `__init__`: Initializes the dataset with the input features `X`.  
    *   `__len__`: Returns the total number of samples in the dataset.  
    *   `__getitem__`: This is the core of the contrastive dataset. For a given index `idx`, it retrieves the original data sample `x`. It then creates two augmented views of `x`, named `x1` and `x2`, by applying the `strong_augment` function defined earlier. The `unsqueeze(0)` and `squeeze(0)` operations are used to handle batch dimensions correctly during augmentation.  
*   `BATCH_SIZE`: Sets the batch size for training.  
*   `train_cl_dataset`: An instance of `ContrastiveHARDataset` is created using the training data `X_train`.  
*   `cl_loader`: A `DataLoader` is created from `train_cl_dataset`. This loader will provide batches of two augmented views (`x1`, `x2`) of the same original data sample during training. Parameters like `shuffle=True` (randomizes batch order), `num_workers` (for parallel data loading), `pin_memory` (for faster GPU transfer), and `drop_last=True` (to drop incomplete batches) are configured.

## 5. Encoder — Deep 1D-CNN

In [ ]:
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = (nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False),
                                    nn.BatchNorm1d(out_ch))
                      if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.skip(x))

class Encoder(nn.Module):
    """
    Input : (B, 9, 128)
    Output: (B, EMBED_DIM)
    """
    def __init__(self, in_channels=9, embed_dim=256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.blocks = nn.Sequential(
            ResBlock1D(64, 64),
            ResBlock1D(64, 128, kernel=5),
            ResBlock1D(128, 128, kernel=5),
            ResBlock1D(128, 256, kernel=3),
            ResBlock1D(256, 256, kernel=3),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Sequential(
            nn.Linear(256, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
        )
        self.embed_dim = embed_dim

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).squeeze(-1)
        return self.proj(x)

This cell defines the `Encoder` network, which is a deep 1D Convolutional Neural Network (CNN) designed to extract features from time-series data.  
*   `ResBlock1D`: This is a building block for the encoder, implementing a residual block structure.  
    *   It consists of two `Conv1d` layers, each followed by `BatchNorm1d` and ReLU activation.  
    *   A `skip` connection allows the input to bypass the convolutional layers and be added to the output, which helps in training deeper networks by mitigating the vanishing gradient problem. If the input and output channel counts differ, a 1x1 convolution is applied to the skip connection to match dimensions.  
*   `Encoder`: This is the main encoder network.  
    *   `__init__`:  
        *   `stem`: The initial layers (`Conv1d`, `BatchNorm1d`, `ReLU`) process the raw input.  
        *   `blocks`: A sequential stack of `ResBlock1D` instances, progressively increasing the number of channels and using different kernel sizes to capture features at various scales.  
        *   `pool`: `AdaptiveAvgPool1d(1)` reduces the spatial dimension of the feature maps to 1, effectively performing global average pooling, which aggregates information across the entire time series.  
        *   `proj`: A linear layer, followed by `BatchNorm1d` and `ReLU`, projects the pooled features to the desired `embed_dim`. This output `(B, EMBED_DIM)` represents the learned embedding of the input time series.  
    *   `forward`: Defines the forward pass through the network, applying the stem, residual blocks, global average pooling, and the final projection.

## 6. Projection Head

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, embed_dim=256, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)

This cell defines the `ProjectionHead` network, which is a shallow Multi-Layer Perceptron (MLP) typically used in contrastive learning frameworks like SimCLR.  
*   `__init__`:  
    *   It consists of a sequential network (`nn.Sequential`) with two linear layers.  
    *   The first linear layer maps the `embed_dim` (output of the `Encoder`) to the same `embed_dim`, followed by `BatchNorm1d` and `ReLU` activation. This acts as a non-linear transformation.  
    *   The second linear layer maps from `embed_dim` to `proj_dim`, producing the final projection.  
*   `forward`: Defines the forward pass, simply applying the `net` (sequential layers) to the input embeddings `x`.  
The purpose of the projection head is to transform the encoder's output into a space where the contrastive loss is applied. According to the SimCLR paper, using a non-linear projection head improves the quality of the learned representations.

## 7. NT-Xent Loss

In [ ]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.T = temperature

    def forward(self, z1, z2):
        B = z1.size(0)
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)
        z  = torch.cat([z1, z2], dim=0)           # (2B, D)
        sim = torch.mm(z, z.T) / self.T            # (2B, 2B)

        # mask out self-similarity
        mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask, float('-inf'))

        # positive indices: i↔i+B and i+B↔i
        pos = torch.cat([torch.arange(B, 2*B, device=z.device),
                         torch.arange(0, B,   device=z.device)])

        loss = F.cross_entropy(sim, pos)
        return loss

This cell implements the NT-Xent (Normalized Temperature-scaled Cross-Entropy) Loss, which is a popular loss function for contrastive learning.  
*   `__init__`: Initializes the loss with a `temperature` parameter. The temperature scales the similarity scores, influencing the sharpness of the distribution. A small temperature makes the distribution sharper, emphasizing larger similarity values.  
*   `forward(self, z1, z2)`:  
    *   `B = z1.size(0)`: Gets the batch size.  
    *   `z1 = F.normalize(z1, dim=1)` and `z2 = F.normalize(z2, dim=1)`: Normalizes the embeddings `z1` and `z2` (the two augmented views from the projection head) to have unit length. This is crucial for cosine similarity.  
    *   `z = torch.cat([z1, z2], dim=0)`: Concatenates the two sets of embeddings along the batch dimension, resulting in a tensor of shape `(2B, D)`, where `D` is `proj_dim`.  
    *   `sim = torch.mm(z, z.T) / self.T`: Computes the cosine similarity matrix between all pairs of embeddings in the concatenated tensor `z`. The result is a `(2B, 2B)` matrix, where `sim[i, j]` is the similarity between `z[i]` and `z[j]`. This is then scaled by the temperature `T`.  
    *   `mask = torch.eye(2 * B, ...)`: Creates an identity matrix used to mask out self-similarity scores (where an embedding is compared with itself, which would always be 1). These values are set to negative infinity (`float('-inf')`) so they don't contribute to the softmax.  
    *   `pos = torch.cat([...])`: Creates a tensor `pos` that identifies the positive pairs. In NT-Xent, for a given embedding `z_i`, its positive pair is `z_{i+B}` (if `i < B`) or `z_{i-B}` (if `i >= B`). These are the augmented views of the same original sample.  
    *   `loss = F.cross_entropy(sim, pos)`: Computes the cross-entropy loss. For each row `i` in the similarity matrix `sim`, the model tries to predict `pos[i]` as the correct positive sample among all other samples in the batch.  
This loss function encourages the model to pull positive pairs (augmented views of the same sample) closer in the embedding space while pushing negative pairs (views of different samples) further apart.

## 8. Build Model, Optimizer and Scheduler

In [ ]:
EMBED_DIM = 256
PROJ_DIM  = 128

encoder    = Encoder(in_channels=9, embed_dim=EMBED_DIM).to(DEVICE)
proj_head  = ProjectionHead(EMBED_DIM, PROJ_DIM).to(DEVICE)
criterion  = NTXentLoss(temperature=0.07)

params = list(encoder.parameters()) + list(proj_head.parameters())
optimizer = torch.optim.AdamW(params, lr=3e-4, weight_decay=1e-4)

# Cosine LR schedule with warmup
EPOCHS_CL  = 100
WARMUP     = 10

def lr_lambda(epoch):
    if epoch < WARMUP:
        return epoch / WARMUP
    progress = (epoch - WARMUP) / (EPOCHS_CL - WARMUP)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

This cell initializes the model components, optimizer, and learning rate scheduler for the contrastive pre-training phase.  
*   `EMBED_DIM`, `PROJ_DIM`: Define the dimensions for the encoder's output and the projection head's output, respectively.  
*   `encoder`: An instance of the `Encoder` network is created and moved to the specified `DEVICE` (CPU or GPU).  
*   `proj_head`: An instance of the `ProjectionHead` network is created and moved to the `DEVICE`.  
*   `criterion`: An instance of the `NTXentLoss` is created with a specified `temperature`.  
*   `params`: A list combining the parameters of both the `encoder` and `proj_head` since both are trained together during contrastive learning.  
*   `optimizer`: An `AdamW` optimizer is configured to update the model parameters. `lr` (learning rate) and `weight_decay` (L2 regularization) are set.  
*   `EPOCHS_CL`, `WARMUP`: Define the total number of epochs for contrastive pre-training and the number of warmup epochs for the learning rate scheduler.  
*   `lr_lambda`: This function defines a cosine annealing learning rate schedule with a linear warmup phase.  
    *   During warmup, the learning rate increases linearly from 0 to the base learning rate.  
    *   After warmup, it decays following a cosine curve, reaching near zero by the end of training. This schedule often helps stabilize training at the beginning and allows for fine-tuning at the end.  
*   `scheduler`: A `LambdaLR` scheduler is created using the `optimizer` and the `lr_lambda` function.

## 9. Contrastive Pre-Training

In [ ]:
print("\n=== Contrastive Pre-Training ===")
cl_losses = []

for epoch in range(1, EPOCHS_CL + 1):
    encoder.train(); proj_head.train()
    epoch_loss = 0

    for x1, x2 in cl_loader:
        x1, x2 = x1.to(DEVICE), x2.to(DEVICE)

        h1, h2 = encoder(x1), encoder(x2)
        z1, z2 = proj_head(h1), proj_head(h2)
        loss   = criterion(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    avg = epoch_loss / len(cl_loader)
    cl_losses.append(avg)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}/{EPOCHS_CL}  Loss: {avg:.4f}  LR: {scheduler.get_last_lr()[0]:.2e}")


=== Contrastive Pre-Training ===


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch    1/100  Loss: 2.9040  LR: 3.00e-05
Epoch   10/100  Loss: 0.3505  LR: 3.00e-04
Epoch   20/100  Loss: 0.2138  LR: 2.91e-04
Epoch   20/100  Loss: 0.2138  LR: 2.91e-04


This cell executes the contrastive pre-training loop.  
*   `print(...)`: Indicates the start of the pre-training phase.  
*   `cl_losses`: An empty list to store the average loss for each epoch.  
*   **Epoch Loop**: The training proceeds for `EPOCHS_CL` epochs.  
    *   `encoder.train(); proj_head.train()`: Sets both the encoder and projection head to training mode.  
    *   `epoch_loss = 0`: Initializes a variable to accumulate loss for the current epoch.  
    *   **Batch Loop**: Iterates through batches from the `cl_loader` (which provides `x1`, `x2` augmented pairs).  
        *   `x1, x2 = x1.to(DEVICE), x2.to(DEVICE)`: Moves the input data to the specified device (GPU/CPU).  
        *   `h1, h2 = encoder(x1), encoder(x2)`: Passes both augmented views through the `encoder` to get their embeddings.  
        *   `z1, z2 = proj_head(h1), proj_head(h2)`: Passes the embeddings through the `ProjectionHead` to get the final projected representations for loss calculation.  
        *   `loss = criterion(z1, z2)`: Calculates the NT-Xent loss between `z1` and `z2`.  
        *   `optimizer.zero_grad()`: Clears previous gradients.  
        *   `loss.backward()`: Computes gradients of the loss with respect to model parameters.  
        *   `torch.nn.utils.clip_grad_norm_`: Clips gradients to prevent exploding gradients.  
        *   `optimizer.step()`: Updates model parameters using the computed gradients.  
        *   `epoch_loss += loss.item()`: Accumulates the loss for the current batch.  
    *   `scheduler.step()`: Updates the learning rate according to the defined schedule.  
    *   `avg = epoch_loss / len(cl_loader)`: Calculates the average loss for the epoch.  
    *   `cl_losses.append(avg)`: Stores the average loss.  
    *   `if epoch % 10 == 0 or epoch == 1`: Prints the epoch number, average loss, and current learning rate periodically to monitor training progress.

## 10. Extract Representations

In [ ]:
print("\n=== Extracting Representations ===")
encoder.eval()

def get_embeddings(X_tensor, bs=512):
    all_h = []
    loader = DataLoader(TensorDataset(X_tensor), batch_size=bs, shuffle=False)
    with torch.no_grad():
        for (x,) in loader:
            all_h.append(encoder(x.to(DEVICE)).cpu())
    return torch.cat(all_h).numpy()

H_train = get_embeddings(X_train)
H_test  = get_embeddings(X_test)

print(f"Train embeddings: {H_train.shape}")
print(f"Test  embeddings: {H_test.shape}")

This cell is responsible for extracting the learned representations (embeddings) from the pre-trained encoder.  
*   `print(...)`: Indicates the start of the representation extraction phase.  
*   `encoder.eval()`: Sets the `encoder` to evaluation mode. This disables dropout layers and batch normalization updates, ensuring consistent behavior for inference.  
*   `get_embeddings(X_tensor, bs=512)`: This helper function takes a tensor of input data and a batch size.  
    *   It creates a `DataLoader` for the input tensor.  
    *   `with torch.no_grad()`: Disables gradient calculation, saving memory and speeding up computations since no backpropagation is needed during inference.  
    *   It iterates through the data, passes each batch `x` through the `encoder` (after moving it to `DEVICE`), moves the resulting embeddings to CPU (`.cpu()`), and appends them to `all_h`.  
    *   Finally, it concatenates all collected embeddings and converts them to a NumPy array.  
*   `H_train = get_embeddings(X_train)`: Extracts embeddings for the training data.  
*   `H_test = get_embeddings(X_test)`: Extracts embeddings for the test data.  
*   The shapes of the extracted training and testing embeddings are printed, showing that each sample is now represented by a `256`-dimensional vector (which is `EMBED_DIM`). These embeddings will be used for downstream tasks like linear evaluation or fine-tuning.

## 11. Linear Evaluation

In [ ]:
print("\n=== Linear Evaluation (Logistic Regression) ===")
# Normalize embeddings before linear eval
sc = StandardScaler()
H_tr_sc = sc.fit_transform(H_train)
H_te_sc = sc.transform(H_test)

clf = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', multi_class='multinomial')
clf.fit(H_tr_sc, y_train)
preds_lr = clf.predict(H_te_sc)
acc_lr   = accuracy_score(y_test, preds_lr)
print(f"Linear Eval Accuracy: {acc_lr*100:.2f}%")

This cell performs linear evaluation using the extracted embeddings. In this common contrastive learning evaluation protocol, the pre-trained encoder's weights are frozen, and a simple linear classifier is trained on top of its features.  
*   `print(...)`: Indicates the start of the linear evaluation phase.  
*   `sc = StandardScaler()`: Initializes another `StandardScaler` to normalize the *embeddings* themselves.  
*   `H_tr_sc = sc.fit_transform(H_train)`: Fits the scaler on the training embeddings and transforms them.  
*   `H_te_sc = sc.transform(H_test)`: Transforms the test embeddings using the scaler fitted on the training embeddings.  
*   `clf = LogisticRegression(...)`: Initializes a `LogisticRegression` classifier from scikit-learn.  
    *   `max_iter=2000`: Sets the maximum number of iterations for the solver.  
    *   `C=1.0`: Inverse of regularization strength; smaller values specify stronger regularization.  
    *   `solver='lbfgs'`: Optimization algorithm.  
    *   `multi_class='multinomial'`: Handles multi-class classification.  
*   `clf.fit(H_tr_sc, y_train)`: Trains the logistic regression model on the scaled training embeddings and their corresponding labels.  
*   `preds_lr = clf.predict(H_te_sc)`: Makes predictions on the scaled test embeddings.  
*   `acc_lr = accuracy_score(y_test, preds_lr)`: Calculates the accuracy of the linear classifier on the test set.  
*   The linear evaluation accuracy is printed. This metric provides insight into how well the pre-trained encoder has learned linearly separable representations.

## 12. Fine-Tuning with Shallow MLP Head

In [ ]:
NUM_CLASSES = 6

class Classifier(nn.Module):
    def __init__(self, encoder, embed_dim, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        h = self.encoder(x)
        return self.head(h)

model = Classifier(encoder, EMBED_DIM, NUM_CLASSES).to(DEVICE)

EPOCHS_FT  = 50
BATCH_FT   = 128

train_ds   = TensorDataset(X_train, y_train_t)
test_ds    = TensorDataset(X_test,  y_test_t)
train_dl   = DataLoader(train_ds, batch_size=BATCH_FT, shuffle=True,  drop_last=True)
test_dl    = DataLoader(test_ds,  batch_size=BATCH_FT, shuffle=False)

opt_ft = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 1e-4},  # lower LR for encoder
    {'params': model.head.parameters(),    'lr': 1e-3},  # higher LR for head
], weight_decay=1e-4)

sched_ft = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=EPOCHS_FT)
ce_loss  = nn.CrossEntropyLoss()

This cell prepares for the fine-tuning phase by defining a `Classifier` model, setting up new data loaders, optimizer, and scheduler.  
*   `NUM_CLASSES = 6`: Defines the number of activity classes in the dataset.  
*   `Classifier` class: This model combines the pre-trained `encoder` with a shallow MLP classification `head`.  
    *   `__init__`: Takes the `encoder`, `embed_dim`, and `num_classes`. It stores the encoder and defines a sequential `head` network with two linear layers, a ReLU activation, and a Dropout layer to prevent overfitting.  
    *   `forward`: Passes the input `x` through the `encoder` to get embeddings (`h`), and then passes `h` through the classification `head` to produce logits for each class.  
*   `model = Classifier(...)`: An instance of the `Classifier` is created, wrapping the pre-trained `encoder`, and moved to the `DEVICE`.  
*   `EPOCHS_FT`, `BATCH_FT`: Define the number of epochs and batch size for fine-tuning.  
*   `train_ds`, `test_ds`: `TensorDataset` objects are created from the `X_train`, `y_train_t` and `X_test`, `y_test_t` tensors.  
*   `train_dl`, `test_dl`: `DataLoader` instances are created for the fine-tuning phase, with `shuffle=True` for training data.  
*   `opt_ft = torch.optim.AdamW([...])`: An `AdamW` optimizer is set up specifically for fine-tuning.  
    *   Crucially, it uses *different learning rates* for the `encoder` parameters (lower LR, `1e-4`) and the new classification `head` parameters (higher LR, `1e-3`). This is a common practice in fine-tuning, as the pre-trained encoder already has good weights and only needs small adjustments, while the new head needs to learn from scratch.  
*   `sched_ft = torch.optim.lr_scheduler.CosineAnnealingLR(...)`: A `CosineAnnealingLR` scheduler is used for fine-tuning, which smoothly decays the learning rate over `EPOCHS_FT`.  
*   `ce_loss = nn.CrossEntropyLoss()`: The standard Cross-Entropy Loss is chosen for classification.

In [ ]:
print("\n=== Fine-Tuning with Shallow MLP Head ===")
best_acc = 0
ft_train_acc, ft_test_acc = [], []

for epoch in range(1, EPOCHS_FT + 1):
    model.train()
    correct = total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss   = ce_loss(logits, yb)
        opt_ft.zero_grad()
        loss.backward()
        opt_ft.step()
        correct += (logits.argmax(1) == yb).sum().item()
        total   += len(yb)
    sched_ft.step()
    tr_acc = correct / total

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += len(yb)
    te_acc = correct / total

    ft_train_acc.append(tr_acc); ft_test_acc.append(te_acc)
    if te_acc > best_acc:
        best_acc = te_acc
        torch.save(model.state_dict(), "best_model.pt")

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS_FT}  "
              f"Train Acc: {tr_acc*100:.2f}%  "
              f"Test Acc:  {te_acc*100:.2f}%")

print(f"\n✅ Best Fine-Tune Test Accuracy: {best_acc*100:.2f}%")

This cell executes the fine-tuning process of the model.  
*   `print(...)`: Indicates the start of the fine-tuning phase.  
*   `best_acc`: Variable to keep track of the highest test accuracy achieved.  
*   `ft_train_acc`, `ft_test_acc`: Lists to store training and testing accuracies per epoch.  
*   **Epoch Loop**:  
    *   `model.train()`: Sets the model to training mode.  
    *   `correct`, `total`: Variables to calculate training accuracy.  
    *   **Training Batch Loop**:  
        *   Moves batch data (`xb`, `yb`) to the `DEVICE`.  
        *   `logits = model(xb)`: Performs a forward pass to get class predictions.  
        *   `loss = ce_loss(logits, yb)`: Calculates the cross-entropy loss.  
        *   `opt_ft.zero_grad()`, `loss.backward()`, `opt_ft.step()`: Standard PyTorch training steps (zero gradients, backpropagation, update weights).  
        *   Updates `correct` and `total` for training accuracy calculation.  
    *   `sched_ft.step()`: Updates the learning rate.  
    *   `tr_acc = correct / total`: Calculates training accuracy.  
    *   `model.eval()`: Sets the model to evaluation mode.  
    *   `with torch.no_grad()`: Disables gradient calculation for inference.  
    *   **Evaluation Batch Loop**:  
        *   Moves batch data to `DEVICE`.  
        *   Calculates `correct` and `total` for test accuracy.  
    *   `te_acc = correct / total`: Calculates test accuracy.  
    *   `ft_train_acc.append(tr_acc); ft_test_acc.append(te_acc)`: Stores accuracies.  
    *   **Model Saving**: If the current `te_acc` is better than `best_acc`, it updates `best_acc` and saves the model's state dictionary to `best_model.pt`.  
    *   Prints training and test accuracies periodically.  
*   Finally, after all epochs, it prints the `best_acc` achieved during fine-tuning.

## 13. Final Classification Report

In [ ]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        all_preds.append(model(xb.to(DEVICE)).argmax(1).cpu())
        all_true.append(yb)
all_preds = torch.cat(all_preds).numpy()
all_true  = torch.cat(all_true).numpy()

LABELS = ["WALKING", "WALKING_UP", "WALKING_DOWN", "SITTING", "STANDING", "LAYING"]
print("\n=== Classification Report ===")
print(classification_report(all_true, all_preds, target_names=LABELS))

This cell loads the best fine-tuned model and generates a comprehensive classification report on the test set.  
*   `model.load_state_dict(torch.load("best_model.pt"))`: Loads the weights of the model that achieved the `best_acc` during fine-tuning.  
*   `model.eval()`: Sets the model to evaluation mode.  
*   `all_preds, all_true`: Empty lists to store all predicted and true labels from the test set.  
*   `with torch.no_grad()`: Disables gradient calculations during inference.  
*   **Test Batch Loop**:  
    *   Iterates through the `test_dl`.  
    *   `model(xb.to(DEVICE)).argmax(1).cpu()`: Gets the model's predictions (class with the highest logit) for each sample in the batch, moves them to CPU.  
    *   `all_preds.append(...)` and `all_true.append(yb)`: Collects the predictions and true labels.  
*   `all_preds = torch.cat(all_preds).numpy()`: Concatenates all predicted tensors and converts them to a NumPy array.  
*   `all_true = torch.cat(all_true).numpy()`: Concatenates all true label tensors and converts them to a NumPy array.  
*   `LABELS`: A list defining the human-readable names for each of the 6 activity classes.  
*   `print(classification_report(...))`: Uses scikit-learn's `classification_report` to print a detailed report, including precision, recall, F1-score, and support for each class, as well as overall accuracy, macro average, and weighted average.

## 14. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cl_losses)
axes[0].set_title("NT-Xent Contrastive Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(True)

axes[1].plot(ft_train_acc, label="Train Acc")
axes[1].plot(ft_test_acc,  label="Test Acc")
axes[1].axhline(best_acc, color='red', linestyle='--', label=f'Best={best_acc*100:.1f}%')
axes[1].set_title("Fine-Tuning Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig("results.png", dpi=150)
plt.show()
print("Saved results.png")

This cell visualizes the training progress of both the contrastive pre-training and the fine-tuning phases.  
*   `fig, axes = plt.subplots(1, 2, figsize=(14, 5))`: Creates a figure with two subplots arranged side-by-side.  
*   **Left Subplot (Contrastive Loss)**:  
    *   `axes[0].plot(cl_losses)`: Plots the stored `cl_losses` (NT-Xent loss) over epochs.  
    *   Sets title, X and Y labels, and enables a grid for readability.  
*   **Right Subplot (Fine-Tuning Accuracy)**:  
    *   `axes[1].plot(ft_train_acc, label="Train Acc")`: Plots the training accuracy during fine-tuning.  
    *   `axes[1].plot(ft_test_acc, label="Test Acc")`: Plots the test accuracy during fine-tuning.  
    *   `axes[1].axhline(best_acc, ...)`: Draws a horizontal dashed red line indicating the `best_acc` achieved on the test set.  
    *   Sets title, X and Y labels, displays a legend, and enables a grid.  
*   `plt.tight_layout()`: Adjusts subplot parameters for a tight layout, preventing labels from overlapping.  
*   `plt.savefig("results.png", dpi=150)`: Saves the generated figure as `results.png` with 150 DPI.  
*   `plt.show()`: Displays the plot.  
This visualization helps in understanding the training dynamics and evaluating the model's performance over time.

## 15. Comparison: Pre-trained vs. From-Scratch Training

This section compares the performance of the pre-trained model (from section 12) with a model trained from scratch (randomly initialized) across different training data sizes. The goal is to demonstrate the benefits of self-supervised pre-training, especially in low-data regimes. For consistency, the full test set (`X_test`) will be used for evaluation, which contains 2947 samples.

In [ ]:
TRAIN_SAMPLE_SIZES = [100, 250, 500, 1000, 2500, 5000, X_train.shape[0]] # X_train.shape[0] is 7352, not 10000

# Prepare constant test DataLoader for evaluation
test_ds_constant = TensorDataset(X_test, y_test_t)
test_dl_constant = DataLoader(test_ds_constant, batch_size=BATCH_FT, shuffle=False)

def run_evaluation_for_comparison(X_train_subset, y_train_subset, test_dl_constant, is_pretrained_encoder,
                                  epochs=EPOCHS_FT, base_lr_encoder=1e-4, base_lr_head=1e-3):
    """Trains and evaluates a Classifier model (either pre-trained or from scratch)."""

    # 1. Initialize model
    temp_encoder = Encoder(in_channels=9, embed_dim=EMBED_DIM).to(DEVICE)
    model_to_train = Classifier(temp_encoder, EMBED_DIM, NUM_CLASSES).to(DEVICE)

    if is_pretrained_encoder:
        # Load the best pre-trained encoder and head
        model_to_train.load_state_dict(torch.load("best_model.pt"))
    else:
        # Model is already randomly initialized at this point for 'from scratch' case
        pass

    # 2. Prepare data loaders for the current subset
    train_ds_subset = TensorDataset(X_train_subset, y_train_subset)
    # Ensure at least one batch is created, if subset size is less than BATCH_FT
    current_batch_size = min(BATCH_FT, len(train_ds_subset))
    if current_batch_size == 0:
        return 0.0 # No data to train on, return 0 accuracy
    train_dl_subset = DataLoader(train_ds_subset, batch_size=current_batch_size, shuffle=True, drop_last=True)

    # 3. Setup optimizer (differential LRs for pretrained, or same LR for scratch)
    if is_pretrained_encoder:
        opt = torch.optim.AdamW([
            {'params': model_to_train.encoder.parameters(), 'lr': base_lr_encoder},
            {'params': model_to_train.head.parameters(),    'lr': base_lr_head},
        ], weight_decay=1e-4)
    else:
        # For training from scratch, typically the learning rate for the whole model is the same
        opt = torch.optim.AdamW(model_to_train.parameters(), lr=base_lr_head, weight_decay=1e-4)

    # 4. Scheduler
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ce_loss = nn.CrossEntropyLoss()

    # 5. Training loop
    for epoch in range(1, epochs + 1):
        model_to_train.train()
        if len(train_dl_subset) == 0: # Handle cases where subset is too small for any batches
            break
        for xb, yb in train_dl_subset:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model_to_train(xb)
            loss   = ce_loss(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()

    # 6. Evaluation
    model_to_train.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_dl_constant:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct += (model_to_train(xb).argmax(1) == yb).sum().item()
            total   += len(yb)
    final_acc = correct / total
    return final_acc

In [ ]:
print("\n=== Running Comparison of Pre-trained vs. From-Scratch ===")

pretrained_accuracies = []
scratch_accuracies = []

for N_samples in TRAIN_SAMPLE_SIZES:
    print(f"\nTraining with {N_samples} samples...")

    # Select subset of training data
    if N_samples <= X_train.shape[0]:
        # Take a contiguous chunk or random sample. For simplicity, take the first N_samples.
        X_train_subset = X_train[:N_samples]
        y_train_subset = y_train_t[:N_samples]
    else:
        # This case should not happen with current TRAIN_SAMPLE_SIZES definition
        print(f"Warning: Requested {N_samples} but only {X_train.shape[0]} available. Using full X_train.")
        X_train_subset = X_train
        y_train_subset = y_train_t

    # Evaluate Pre-trained Model
    print("  -> Evaluating Pre-trained model...")
    acc_pretrained = run_evaluation_for_comparison(
        X_train_subset, y_train_subset, test_dl_constant, is_pretrained_encoder=True,
        epochs=EPOCHS_FT # Using same epochs as original fine-tuning
    )
    pretrained_accuracies.append(acc_pretrained)
    print(f"     Pre-trained Model Test Accuracy ({N_samples} samples): {acc_pretrained*100:.2f}%")

    # Evaluate From-Scratch Model
    print("  -> Evaluating From-Scratch model...")
    acc_scratch = run_evaluation_for_comparison(
        X_train_subset, y_train_subset, test_dl_constant, is_pretrained_encoder=False,
        epochs=EPOCHS_FT # Using same epochs as original fine-tuning
    )
    scratch_accuracies.append(acc_scratch)
    print(f"     From-Scratch Model Test Accuracy ({N_samples} samples): {acc_scratch*100:.2f}%")

print("\nComparison complete.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(TRAIN_SAMPLE_SIZES, [a * 100 for a in pretrained_accuracies], marker='o', label='Pre-trained Model')
plt.plot(TRAIN_SAMPLE_SIZES, [a * 100 for a in scratch_accuracies], marker='x', label='From-Scratch Model')

plt.title('Test Accuracy vs. Training Data Size: Pre-trained vs. From-Scratch')
plt.xlabel('Number of Training Samples')
plt.ylabel('Test Accuracy (%)')
plt.xticks(TRAIN_SAMPLE_SIZES)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

print("\nThis plot illustrates the significant advantage of pre-training, especially when the amount of labeled training data is limited. The pre-trained model consistently outperforms the randomly initialized model across various training data sizes, highlighting the effectiveness of self-supervised learning for learning good representations.")

## 16. Visualization of Embeddings with PCA

To understand how well the encoder has learned to separate different activity classes, we can visualize the high-dimensional embeddings (`H_train`) in a 2D space using Principal Component Analysis (PCA). Each point in the scatter plot will represent a training sample, colored according to its true activity label. A good separation of clusters by color indicates that the model has learned discriminative features.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

print("\n=== Visualizing Embeddings with PCA ===")

# Perform PCA to reduce embeddings to 2 dimensions
pca = PCA(n_components=2)
H_train_pca = pca.fit_transform(H_train)

# Create a DataFrame for easier plotting with seaborn
import pandas as pd
pca_df = pd.DataFrame({
    'PC1': H_train_pca[:, 0],
    'PC2': H_train_pca[:, 1],
    'Activity': [LABELS[int(y)] for y in y_train] # Map numerical labels to readable names
})

plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='PC1', y='PC2', hue='Activity', data=pca_df,
    palette='viridis', s=30, alpha=0.7, legend='full'
)
plt.title('PCA of Training Embeddings (Colored by Activity)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("The scatter plot shows the training data embeddings projected onto their two main principal components. Distinct clusters for different activities suggest that the pre-trained encoder has learned effective representations for distinguishing between human activities.")